# Export MGXS from OpenMC model

The following notebook uses a previously built OpenMC `model.xml` and keeps the transport-to-constants workflow in reusable Python code inside `mgxs_export.py`.

### Features
- Use a previously built reactor `model.xml` from `build/`
- Select a multigroup export
- Choose either a supercell core domain or resolved core substructure domains
- Export separate constants for the core, moderator region, and reflector region
- Write machine-readable JSON and CSV outputs under `build/{reactor_type}/mgxs_export/outputs`

### Contract
- Run with the `openmc` conda environment and configured OpenMC nuclear data
- Prefer reusable Python modules over copy-pasted notebook logic

### Note
The delayed neutron beta values exported here are delayed neutron fractions from MGXS tallies. They are a useful starting point for point kinetics, but they are not the same as adjoint-weighted `beta_eff` from an IFP-style calculation. 

# Results

**A. Original coarse geometry (few homogenized regions)**

Setup as a "supercell" with: 
- fuel lumped,
- moderator lumped,
- reflector lumped.

| Groups | MG (k_{eff}) | Δρ [pcm] |
|----------|:-------------:|------:|
2 |	0.991	| ~ -1000 pcm
8 |	0.987	| ~ -1400 pcm
16	| 0.9866	| ~ -1450 pcm

This shows: strong underprediction with worsening with more groups and a clear homogenization failure. The dominant error source here was spatial homogenization, not energy resolution.

A new refined geometry models the MGXS for every openmc cell.

**B. Refined geometry + P0 treatment (transport corrected isotropic scattering)**
| Groups | MG (k_{eff}) | Δρ [pcm] |
| ------ | ----------------: | -------: |
| 1      |          1.020588 |    +1908 |
| 2      |          1.009235 |     +806 |
| 4      |          1.005379 |     +426 |
| 8      |          1.002756 |     +166 |
| 16     |          1.001754 |      +66 |
| 25     |          0.999614 |     -148 |
| 40     |          0.998643 |     -245 |
| 70     |          0.998703 |     -239 |

Now the behavior became physically meaningful:
- few groups overpredict,
- intermediate groups best,
- many groups slightly low.

This is typical MG behavior but has room for improvement to represent isotropic scattering


**C. Refined geometry + P1 scattering** 
| Groups | MG (k_{eff}) | Δρ [pcm] |
| ------ | ----------------: | -------: |
| 1      |          1.022087 |    +2052 |
| 2      |          1.011829 |    +1060 |
| 4      |          1.008152 |     +700 |
| 8      |          1.004322 |     +322 |
| 16     |          1.005241 |     +413 |
| 25     |          1.002186 |     +109 |
| 40     |          1.001733 |      +64 |
| 70     |          1.001440 |      +35 |

P1 improves high-group transport consistency and angular leakage representation.

Residual bias became ≲50 pcm at high groups, which is on the same level as statistical uncertainty from the Monte Carlo model used.

**D. Refined geometry + P3 scattering**
| Groups | MG (k_{eff}) | Δρ [pcm] |
| ------ | ----------------: | -------: |
| 1      |          1.021338 |    +1980 |
| 2      |          1.012732 |    +1148 |
| 4      |          1.007359 |     +622 |
| 8      |          1.004988 |     +388 |
| 16     |          1.005290 |     +417 |
| 25     |          1.002715 |     +162 |
| 40     |          1.000717 |      -37 |
| 70     |          1.001343 |      +25 |

P3 produces the best asymptotic convergence at a slightly higher computational cost. This shows that angular scattering is a dominant error source when using multi-group transport.

Overall the biggest improvement comes from a finer geometry because neutron spectra varies strongly spatially in a thermal system and a single homogenized region doesn't preserve the leakage coupling. We also see that few group condensation gives worse results, coming from the smearing of resonance absorption, epithermal slowing-down and thermal-leakage coupling. Finally as we increase the legendre polynomial orders we see that results get better, this is because hydrogen scattering is strongly anisotropic so without angular moments neutrons random-walk too diffusively and leakage becomes too large, thus k_{P0} \lt k_{CE}. P1/P3 do a sufficient job of preserving forward scattering, the angular transport behaviour and overall reduce artifical leakage.

In [1]:
import importlib
from dataclasses import asdict
from pathlib import Path
from pprint import pprint
import sys

import openmc

import mgxs_export
from ploting import resolve_openmc_exec

importlib.reload(mgxs_export)

from mgxs_export import (
    MGXSExportConfig,
    build_directories,
    cross_sections_path,
    default_openmc_threads,
    domain_definitions_for_core_modeling,
    domain_mapping_from_definitions,
    energy_group_edges_for_group_count,
    inspect_model_source,
    legendre_order_for_moment,
    publish_group_sweep,
    resolve_model_xml_path,
    run_openmc_mg_validation,
    run_mgxs_export,
    )

## Select the built core model and group structure

Point `MODEL_SOURCE_DIR` at a build directory that already contains a `model.xml`. Set `GROUP_COUNT` to choose the exported energy-group structure.

Set `CORE_MODELING` to:
- `"supercell"` to keep the current approach (`core -> fuel_element`)
- `"resolved"` to split the core into `fuel_ring_*` domains plus `central_moderator_channel` when present

Set `SCATTERING_MOMENT` to choose the Legendre order used for scattering moments in both the CE setup and MGXS library:
- `"P0"` (zeroth-order)
- `"P1"` (first-order)
- `"P3"` (third-order)
- `"P5"` (fifth order)

This does NOT make a difference on a MG diffusion solver but will on MG MonteCarlo or SP3, SN determinsitic codes.

Both core-modeling modes keep `moderator -> d2o_tank` and `reflector -> h2o_tank`.

The library utilizes OpenMC's built-in CASMO group structures (energy groups = [2,4,8,16,25,40,70]).

The default MGXS tally list also includes the `nu-scatter matrix` needed if you want to turn the exported constants back into an OpenMC multigroup calculation for a `k_eff` validation run.

This notebook does not rebuild the core. If `model.xml` is missing, the notebook raises an error so you can build the transport model first.

In [3]:
MODEL_SOURCE_DIR = Path("build") / "concentric"  # edit this path to point at the built model directory
CORE_MODELING = "resolved"                      # choose "supercell" or "resolved"
GROUP_COUNT = 4                                 # choose 1, 2, 4, 8, 16, 25, 40, or 70 energy groups
SCATTERING_MOMENT = "P0"                        # choose "P0", "P1", or "P3"

MODEL_XML_PATH = resolve_model_xml_path(MODEL_SOURCE_DIR)
DOMAIN_DEFINITIONS = domain_definitions_for_core_modeling(
    MODEL_XML_PATH,
    core_modeling=CORE_MODELING,
)
ENERGY_GROUP_EDGES_EV = energy_group_edges_for_group_count(GROUP_COUNT)
LEGENDRE_ORDER = legendre_order_for_moment(SCATTERING_MOMENT)

EXPORT_CONFIG = MGXSExportConfig(
    particles=200000,
    batches=100,
    inactive=10,
    domain_definitions=DOMAIN_DEFINITIONS,
    energy_group_edges_ev=ENERGY_GROUP_EDGES_EV,
    legendre_order=LEGENDRE_ORDER,
    scatter_correction=None,
    validation_scatter_correction="P0",
    )

EXPORT_PATHS = build_directories(MODEL_XML_PATH)
OPENMC_EXEC = resolve_openmc_exec()
OPENMC_THREADS = default_openmc_threads()
CROSS_SECTIONS = cross_sections_path()
MODEL_REPORT = inspect_model_source(
    MODEL_XML_PATH,
    domain_definitions=EXPORT_CONFIG.domain_definitions,
    )

NOTEBOOK_REPORT = {
    "python_executable": sys.executable,
    "openmc_version": openmc.__version__,
    "openmc_exec": OPENMC_EXEC,
    "openmc_threads": OPENMC_THREADS,
    "cross_sections": CROSS_SECTIONS,
    "model_source_dir": str(MODEL_SOURCE_DIR),
    "model_xml_path": str(MODEL_XML_PATH),
    "core_modeling": CORE_MODELING,
    "group_count": GROUP_COUNT,
    "scattering_moment": SCATTERING_MOMENT,
    "legendre_order": LEGENDRE_ORDER,
    "energy_group_edges_ev": list(ENERGY_GROUP_EDGES_EV),
    "domain_count": len(DOMAIN_DEFINITIONS),
    "domain_mapping": domain_mapping_from_definitions(DOMAIN_DEFINITIONS),
    "model_report": MODEL_REPORT,
    "export_config": asdict(EXPORT_CONFIG),
    "output_directory": str(EXPORT_PATHS["output_dir"]),
}

pprint(NOTEBOOK_REPORT, sort_dicts=False)

{'python_executable': '/home/pablo/miniconda3/envs/openmc/bin/python',
 'openmc_version': '0.15.3',
 'openmc_exec': '/home/pablo/miniconda3/envs/openmc/bin/openmc',
 'openmc_threads': 19,
 'cross_sections': PosixPath('/home/pablo/openmc/data/endfb-viii.1-hdf5/cross_sections.xml'),
 'model_source_dir': 'build/concentric',
 'model_xml_path': '/home/pablo/modelica/repo-codex-a/openmc/build/concentric/model.xml',
 'core_modeling': 'resolved',
 'group_count': 4,
 'scattering_moment': 'P0',
 'legendre_order': 0,
 'energy_group_edges_ev': [0.0, 0.625, 5530.0, 821000.0, 20000000.0],
 'domain_count': 12,
 'domain_mapping': {'core_central_moderator_channel': 'central_moderator_channel',
                    'core_fuel_ring_1': 'fuel_ring_1',
                    'core_fuel_ring_2': 'fuel_ring_2',
                    'core_fuel_ring_3': 'fuel_ring_3',
                    'core_fuel_ring_4': 'fuel_ring_4',
                    'core_fuel_ring_5': 'fuel_ring_5',
                    'core_fuel_ring_6':

## Run the export

This cell loads the already-built `model.xml` from the selected build directory, attaches MGXS tallies and delayed-neutron tallies to the configured regions, and writes JSON plus CSV outputs into the export directory. The tally set also includes the scatter information needed for the OpenMC multigroup validation section below.

In [4]:
if not CROSS_SECTIONS:
    results = None
    print(
        "Set openmc.config['cross_sections'] or OPENMC_CROSS_SECTIONS and ensure the openmc conda environment is active before running this cell."
    )
else:
    results = run_mgxs_export(
        model_path_or_dir=MODEL_XML_PATH,
        config=EXPORT_CONFIG,
        export_dir=EXPORT_PATHS["root_dir"],
        threads=OPENMC_THREADS,
        openmc_exec=OPENMC_EXEC,
    )
    print(f"Statepoint: {results['statepoint_path']}")
    print(f"JSON export: {results['files']['json']}")
    print(f"Group constants CSV: {results['files']['group_constants_csv']}")
    print(f"Delayed neutron CSV: {results['files']['delayed_neutrons_csv']}")

/home/pablo/miniconda3/envs/openmc/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Filter instance already exists with id=1649.
  warn(msg, IDWarning)
/home/pablo/miniconda3/envs/openmc/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Filter instance already exists with id=4.
  warn(msg, IDWarning)
/home/pablo/miniconda3/envs/openmc/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Filter instance already exists with id=7.
  warn(msg, IDWarning)
/home/pablo/miniconda3/envs/openmc/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Filter instance already exists with id=21.
  warn(msg, IDWarning)
/home/pablo/miniconda3/envs/openmc/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Filter instance already exists with id=1652.
  warn(msg, IDWarning)
/home/pablo/miniconda3/envs/openmc/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Filter instance already exists with id=1814.
  war

                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

/home/pablo/miniconda3/envs/openmc/lib/python3.13/site-packages/openmc/tallies.py:3009: RuntimeWarning: divide by zero encountered in reciprocal
  new_tally._mean = self._mean ** power
/home/pablo/miniconda3/envs/openmc/lib/python3.13/site-packages/openmc/tallies.py:3010: RuntimeWarning: invalid value encountered in divide
  self_rel_err = self.std_dev / self.mean


## Summarize the exported constants

Use this after the export cell to inspect the transport and delayed-neutron constants written for each exported region.

In [5]:
if results is None:
    print("Run the export cell first.")
else:
    run = results["run"]
    group_count = len(results["config"]["energy_group_edges_ev"]) - 1
    print(f"k_eff = {run['keff']['mean']:.6f} +/- {run['keff']['std_dev']:.6f}")
    print(f"reactivity = {run['reactivity_pcm']:.1f} pcm")
    print(f"group count = {group_count}")

    generation_time = run.get("generation_time_s")
    if generation_time is not None:
        print(
            "generation time = "
            f"{generation_time['mean']:.6e} +/- {generation_time['std_dev']:.6e} s"
        )

    for domain_label, domain_payload in results["domains"].items():
        delayed = domain_payload["delayed_neutrons"]
        print(f"\n[{domain_label}] cell={domain_payload['domain']['name']}")
        print(f"beta_total = {delayed['beta_total']:.6e}")
        print(f"beta_total_by_energy_group = {delayed['beta_total_by_energy_group']}")
        if delayed["beta_weighted_decay_rate_per_s"] is not None:
            print(
                "beta-weighted decay rate = "
                f"{delayed['beta_weighted_decay_rate_per_s']:.6e} 1/s"
            )

        print("Delayed groups")
        beta_by_delayed_group = delayed["beta_by_delayed_group"]
        beta_totals = delayed["beta_total_by_delayed_group"]
        decay_rates = delayed["decay_rate_per_s_by_delayed_group"]
        for delayed_group_index, delayed_group in enumerate(results["config"]["delayed_groups"]):
            beta_value = beta_by_delayed_group[delayed_group_index]
            print(
                f"group {delayed_group}: total_beta={beta_totals[delayed_group_index]:.6e}, "
                f"beta={beta_value}, lambda={decay_rates[delayed_group_index]:.6e} 1/s"
            )

        print("Group constants")
        for xs_type, payload in domain_payload["group_constants"].items():
            print(f"{xs_type:>22}: mean={payload['mean']} std_dev={payload['std_dev']}")

k_eff = 1.001079 +/- 0.000593
reactivity = 107.8 pcm
group count = 4

[core_central_moderator_channel] cell=central_moderator_channel
beta_total = 0.000000e+00
beta_total_by_energy_group = [0.0]
Delayed groups
group 1: total_beta=0.000000e+00, beta=0.0, lambda=1.334432e-02 1/s
group 2: total_beta=0.000000e+00, beta=0.0, lambda=3.267764e-02 1/s
group 3: total_beta=0.000000e+00, beta=0.0, lambda=1.209148e-01 1/s
group 4: total_beta=0.000000e+00, beta=0.0, lambda=3.042026e-01 1/s
group 5: total_beta=0.000000e+00, beta=0.0, lambda=8.554143e-01 1/s
group 6: total_beta=0.000000e+00, beta=0.0, lambda=2.872955e+00 1/s
Group constants
                 total: mean=[0.23412122323915885, 0.34014068879570414, 0.3521979193969983, 0.4461906822399561] std_dev=[0.003209700254552936, 0.0032506504817403194, 0.0033961107725837135, 0.0034369439056101433]
             transport: mean=[0.16821269600750066, 0.283255582542705, 0.2723203883479883, 0.39500890174694514] std_dev=[0.0034806217212853085, 0.003383927

## Validation: run OpenMC in multigroup mode

This section rebuilds an OpenMC multigroup library from the just-generated MGXS statepoint, writes an `mgxs.h5`, and runs an eigenvalue calculation with `settings.energy_mode = 'multi-group'`.

Use this as a transport-side validation of how much `k_eff` changes when the same geometry is solved with OpenMC's multigroup mode instead of the original continuous-energy run. Re-run the export cell first if you changed the tally configuration above.

In [6]:
MG_VALIDATION_DIR = EXPORT_PATHS["root_dir"] / "mg_mode_validation"

if results is None:
    mg_validation = None
    print("Run the export cell first.")
else:
    mg_validation = run_openmc_mg_validation(
        statepoint_path=results["statepoint_path"],
        model_path_or_dir=MODEL_XML_PATH,
        config=EXPORT_CONFIG,
        output_dir=MG_VALIDATION_DIR,
        particles=20000,
        batches=100,
        inactive=10,
        threads=OPENMC_THREADS,
        openmc_exec=OPENMC_EXEC,
    )

    ce_keff = results["run"]["keff"]
    mg_keff = mg_validation["run"]["keff"]
    delta_k = mg_keff["mean"] - ce_keff["mean"]
    delta_pcm = ((mg_keff["mean"] - ce_keff["mean"]) / mg_keff["mean"]) * 1.0e5

    print(f"Group count                = {GROUP_COUNT}")
    print(
        "Continuous-energy k_eff   = "
        f"{ce_keff['mean']:.6f} +/- {ce_keff['std_dev']:.6f}"
    )
    print(
        "OpenMC multi-group k_eff   = "
        f"{mg_keff['mean']:.6f} +/- {mg_keff['std_dev']:.6f}"
    )
    print(f"Delta k                    = {delta_k:+.6f}")
    print(f"Approx. delta reactivity   = {delta_pcm:+.1f} pcm")
    print(f"MGXS HDF5                  = {mg_validation['mgxs_hdf5_path']}")
    print(f"MG validation statepoint   = {mg_validation['statepoint_path']}")

/home/pablo/miniconda3/envs/openmc/lib/python3.13/site-packages/openmc/mgxs_library.py:1640: RuntimeWarning: invalid value encountered in divide
  self._multiplicity_matrix[i] = np.divide(nuscatt, scatt)
/home/pablo/miniconda3/envs/openmc/lib/python3.13/site-packages/openmc/mgxs/library.py:1501: UserWarning: If the library domain includes a lattice or universe cell in conjunction with a consituent cell of that lattice/universe, the multi-group simulation will fail
  warn('If the library domain includes a lattice or universe cell '


                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

# Multigroup Sweep Publication

Raw OpenMC runs remain under ignored `build/`. Set `RUN_GROUP_SWEEP` only when transport calculations must be regenerated. Set `PUBLISH_REFERENCE_SWEEP` to validate and promote only `model.xml`, `mgxs_constants.json`, and a checksum manifest into tracked `openmc/reference_data/concentric/group_sweep`.


In [4]:
RUN_GROUP_SWEEP = True
PUBLISH_REFERENCE_SWEEP = True

# SWEEP_GROUP_COUNTS = tuple(mgxs_export.SUPPORTED_GROUP_COUNTS)
SWEEP_GROUP_COUNTS = (1, 2, 4, 8, 16)
RAW_SWEEP_EXPORT_ROOT = EXPORT_PATHS["root_dir"] / "group_sweep"
OPENMC_ROOT = Path(mgxs_export.__file__).resolve().parent
REFERENCE_SWEEP_ROOT = (
    OPENMC_ROOT
    / "reference_data"
    / "concentric"
    / "group_sweep"
)
GROUP_SWEEP_RESULTS = []

if not CROSS_SECTIONS and RUN_GROUP_SWEEP:
    print(
        "Set openmc.config['cross_sections'] or OPENMC_CROSS_SECTIONS and ensure "
        "the openmc conda environment is active before running this cell."
    )
elif RUN_GROUP_SWEEP:
    print(
        f"Running full workflow for groups: {SWEEP_GROUP_COUNTS} "
        f"with scattering moment P{EXPORT_CONFIG.legendre_order}"
    )

    for group_count in SWEEP_GROUP_COUNTS:
        group_export_dir = RAW_SWEEP_EXPORT_ROOT / f"group_{group_count}"
        group_edges = energy_group_edges_for_group_count(group_count)
        group_config = MGXSExportConfig(
            particles=EXPORT_CONFIG.particles,
            batches=EXPORT_CONFIG.batches,
            inactive=EXPORT_CONFIG.inactive,
            domain_definitions=DOMAIN_DEFINITIONS,
            energy_group_edges_ev=group_edges,
            mgxs_types=EXPORT_CONFIG.mgxs_types,
            delayed_groups=EXPORT_CONFIG.delayed_groups,
            legendre_order=EXPORT_CONFIG.legendre_order,
            scatter_correction=EXPORT_CONFIG.scatter_correction,
            validation_scatter_correction=EXPORT_CONFIG.validation_scatter_correction,
            kinetics_group_count=EXPORT_CONFIG.kinetics_group_count,
            reference_power_mesh_radial_bins=EXPORT_CONFIG.reference_power_mesh_radial_bins,
            reference_power_mesh_axial_bins=EXPORT_CONFIG.reference_power_mesh_axial_bins,
        )

        print(f"\n=== {group_count}-group export ===")
        group_export = run_mgxs_export(
            model_path_or_dir=MODEL_XML_PATH,
            config=group_config,
            export_dir=group_export_dir,
            threads=OPENMC_THREADS,
            openmc_exec=OPENMC_EXEC,
        )

        print(f"=== {group_count}-group validation ===")
        group_validation = run_openmc_mg_validation(
            statepoint_path=group_export["statepoint_path"],
            model_path_or_dir=MODEL_XML_PATH,
            config=group_config,
            output_dir=group_export_dir / "mg_mode_validation",
            particles=16000,
            batches=50,
            inactive=10,
            threads=OPENMC_THREADS,
            openmc_exec=OPENMC_EXEC,
        )

        ce_keff = group_export["run"]["keff"]
        mg_keff = group_validation["run"]["keff"]
        rho_ce_pcm = ((ce_keff["mean"] - 1.0) / ce_keff["mean"]) * 1.0e5
        rho_mg_pcm = ((mg_keff["mean"] - 1.0) / mg_keff["mean"]) * 1.0e5
        delta_rho_pcm = rho_mg_pcm - rho_ce_pcm

        row = {
            "group_count": group_count,
            "scattering_moment": f"P{group_config.legendre_order}",
            "ce_keff_mean": ce_keff["mean"],
            "ce_keff_std_dev": ce_keff["std_dev"],
            "mg_keff_mean": mg_keff["mean"],
            "mg_keff_std_dev": mg_keff["std_dev"],
            "rho_ce_pcm": rho_ce_pcm,
            "rho_mg_pcm": rho_mg_pcm,
            "delta_rho_pcm": delta_rho_pcm,
            "ce_statepoint_path": group_export["statepoint_path"],
            "mg_statepoint_path": group_validation["statepoint_path"],
        }
        GROUP_SWEEP_RESULTS.append(row)

        print(
            f"{group_count:>3} groups: CE k_eff={ce_keff['mean']:.6f} +/- {ce_keff['std_dev']:.6f}, "
            f"MG k_eff={mg_keff['mean']:.6f} +/- {mg_keff['std_dev']:.6f}, "
            f"delta rho={delta_rho_pcm:+.1f} pcm"
        )

    print("\nDelta reactivity vs CE (rho_MG - rho_CE):")
    print(f"Scattering moment: P{EXPORT_CONFIG.legendre_order}")
    print(f"{'groups':>8} {'CE k_eff':>18} {'MG k_eff':>18} {'Delta rho [pcm]':>18}")
    for row in GROUP_SWEEP_RESULTS:
        print(
            f"{row['group_count']:>8d} "
            f"{row['ce_keff_mean']:.6f} +/- {row['ce_keff_std_dev']:.6f} "
            f"{row['mg_keff_mean']:.6f} +/- {row['mg_keff_std_dev']:.6f} "
            f"{row['delta_rho_pcm']:+.1f}"
        )


if PUBLISH_REFERENCE_SWEEP:
    published = publish_group_sweep(
        RAW_SWEEP_EXPORT_ROOT,
        REFERENCE_SWEEP_ROOT,
        group_counts=SWEEP_GROUP_COUNTS,
    )
    print(f"Published reference sweep: {published['root']}")
    print(f"Manifest: {published['manifest']}")
elif not RUN_GROUP_SWEEP:
    print("Sweep execution and publication are disabled.")


Running full workflow for groups: (1, 2, 4, 8, 16) with scattering moment P0

=== 1-group export ===


/home/pablo/miniconda3/envs/openmc/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Filter instance already exists with id=1707.
  warn(msg, IDWarning)
/home/pablo/miniconda3/envs/openmc/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Filter instance already exists with id=2.
  warn(msg, IDWarning)
/home/pablo/miniconda3/envs/openmc/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Filter instance already exists with id=7.
  warn(msg, IDWarning)
/home/pablo/miniconda3/envs/openmc/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Filter instance already exists with id=1652.
  warn(msg, IDWarning)
/home/pablo/miniconda3/envs/openmc/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Filter instance already exists with id=1815.
  warn(msg, IDWarning)


                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

/home/pablo/miniconda3/envs/openmc/lib/python3.13/site-packages/openmc/mgxs/library.py:1501: UserWarning: If the library domain includes a lattice or universe cell in conjunction with a consituent cell of that lattice/universe, the multi-group simulation will fail
  warn('If the library domain includes a lattice or universe cell '


                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

...

                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

/home/pablo/miniconda3/envs/openmc/lib/python3.13/site-packages/openmc/tallies.py:3009: RuntimeWarning: divide by zero encountered in reciprocal
  new_tally._mean = self._mean ** power
/home/pablo/miniconda3/envs/openmc/lib/python3.13/site-packages/openmc/tallies.py:3010: RuntimeWarning: invalid value encountered in divide
  self_rel_err = self.std_dev / self.mean


=== 16-group validation ===


/home/pablo/miniconda3/envs/openmc/lib/python3.13/site-packages/openmc/mgxs_library.py:1640: RuntimeWarning: invalid value encountered in divide
  self._multiplicity_matrix[i] = np.divide(nuscatt, scatt)
/home/pablo/miniconda3/envs/openmc/lib/python3.13/site-packages/openmc/mgxs/library.py:1501: UserWarning: If the library domain includes a lattice or universe cell in conjunction with a consituent cell of that lattice/universe, the multi-group simulation will fail
  warn('If the library domain includes a lattice or universe cell '


                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################